https://velog.io/@kungsboy/%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D-05-1.%EA%B2%B0%EC%A0%95%ED%8A%B8%EB%A6%AC-%EA%B8%B0%EB%B3%B8-%EC%98%88%EC%A0%9C

| 날씨   | 습도   | 바람   |
| ---- | ---- | ---- |
| 0 맑음 | 1 높음 | 0 약함 |
| 0 맑음 | 0 낮음 | 0 약함 |
| 1 흐림 | 1 높음 | 1 강함 |
| 2 비  | 1 높음 | 0 약함 |


In [1]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
import pandas as pd

# 날씨 데이터 (숫자 인코딩)
# 0: 맑음, 1: 흐림, 2: 비
# 0: 낮음/약함, 1: 높음/강함

# plt.rc('font', family='NanumGothic')

# 데이터프레임 생성
df = pd.DataFrame({
    "날씨": [0, 0, 1, 2],
    "습도": [1, 0, 1, 1],
    "바람": [0, 0, 1, 0],
    "테니스": [0, 1, 1, 0]
})


# 모델 생성
model = DecisionTreeClassifier(random_state=42)

X = df[["날씨", "습도", "바람"]]
y = df["테니스"]

# 학습
model.fit(X, y)

# 예측
print(model.predict([[0, 1, 0]]))
# 날씨 = 맑음
# 습도 = 높음
# 바람 = 약함

# wind <= 0.5
# ↓
# True
# ↓
# 왼쪽 이동
# ↓
# 최종 예측 = No

In [2]:
# 🌳 트리 시각화
# 시각화
plt.figure(figsize=(10, 6))

plot_tree(
    model,
    # feature_names = X.columns,      # 컬럼명 표시
    feature_names = ["weather", "moisture", "wind"],
    class_names=["No", "Yes"],    # 0,1 이름 변경
    filled=True,                  # 색상
    rounded=True,                 # 둥근 모서리
    fontsize=12
)

plt.show()

# 내부동작 순서
# 트리는 내부적으로 이렇게 계산

# 날씨로 나누면 얼마나 잘 분리되지?
# ↓

# 습도로 나누면?
# ↓

# 바람으로 나누면?

# 그리고 불순도(Gini)가 가장 많이 감소하는 변수를 Root로 선택

# 결정트리의 지니 계수(Gini Index)

## 1. 지니 계수란

지니 계수(Gini Index)는 결정트리에서 **데이터가 얼마나 섞여 있는지(불순도, Impurity)** 를 측정하는 지표이다.

즉,

* 여러 클래스가 섞여 있으면 → 지니 값 증가
* 하나의 클래스만 존재하면 → 지니 값 감소

결정트리는 **지니 계수를 가장 많이 감소시키는 방향으로 분기한다.**

---

# 2. 직관적 이해

## 경우 ① 데이터가 섞여 있는 경우

| 결과  |
| --- |
| 합격  |
| 불합격 |
| 합격  |
| 불합격 |

분포

* 합격 = 2
* 불합격 = 2

→ 여러 클래스가 섞임
→ 지니 계수 큼

---

## 경우 ② 완전히 분리된 경우

| 결과 |
| -- |
| 합격 |
| 합격 |
| 합격 |
| 합격 |

분포

* 합격 = 4
* 불합격 = 0

→ 섞여 있지 않음
→ Gini = 0

---

# 3. 지니 계수 공식

$$
Gini = 1 - \sum p_i^2
$$

설명

* $p_i$ : 각 클래스 비율
* 클래스 비율 제곱의 합을 1에서 뺀 값

---

# 4. 계산 예제

데이터

| 결과  |
| --- |
| No  |
| No  |
| Yes |
| Yes |

비율

* No = 2/4 = 0.5
* Yes = 2/4 = 0.5

공식 적용

$$
Gini
====

1-(0.5^2+0.5^2)
$$

계산

$$
=1-(0.25+0.25)
$$

결과

$$
=0.5
$$

해석

→ 데이터가 많이 섞여 있음

---

# 5. 결정트리에서 지니 계수 사용

예시 데이터

| 날씨 | 테니스 |
| -- | --- |
| 맑음 | No  |
| 맑음 | Yes |
| 흐림 | Yes |
| 비  | No  |

현재 상태

* No = 2
* Yes = 2

현재 지니

$$
Gini
====

# 1-(0.5^2+0.5^2)

0.5
$$

---

## 습도로 분리

### 왼쪽 노드

| 결과  |
| --- |
| Yes |

$$
Gini=0
$$

완전히 분리됨

---

### 오른쪽 노드

| 결과  |
| --- |
| No  |
| Yes |
| No  |

비율

* No = 2/3
* Yes = 1/3

계산

$$
Gini
====

1-
\left(\frac{2}{3}\right)^2
--------------------------

\left(\frac{1}{3}\right)^2
$$

# $$

1-0.444-0.111
$$

# $$

0.445
$$

---

분리 전

$$
0.5
$$

↓

분리 후

$$
0 + 0.445
$$

불순도 감소

↓

좋은 분할

---

# 6. 지니 계수 해석 기준

| Gini | 의미    |
| ---: | ----- |
|    0 | 완전 분리 |
|  0.1 | 매우 깨끗 |
|  0.3 | 약간 섞임 |
|  0.5 | 많이 섞임 |

※ 이진 분류에서는 보통 최대값이 0.5

---

# 7. 파이썬 코드 예제

```python
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    criterion="gini"
)

model.fit(X, y)
```

예시 출력

```text
gini = 0.375
samples = 8
value = [6,2]
```

해석

* 데이터 수: 8개
* No: 6개
* Yes: 2개
* 비교적 잘 분리된 상태

---

# 8. 정리

지니 계수

↓

노드 안 데이터가 얼마나 섞여 있는지 측정

↓

값이 작을수록 좋음

↓

0이면 완전 분리

↓

결정트리는 지니를 줄이는 방향으로 분기


In [ ]:
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.tree import plot_tree
import matplotlib.pyplot as plt


# 1 데이터 로드
iris = load_iris()

X = iris.data
y = iris.target


# 2 모델 생성
dt = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)


# 3 데이터 분할
train_x, test_x, train_y, test_y = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# 4 학습
dt.fit(train_x, train_y)


# 5 예측
train_pred = dt.predict(train_x)
test_pred = dt.predict(test_x)


# 6 성능 평가
train_acc = accuracy_score(train_y, train_pred)
test_acc = accuracy_score(test_y, test_pred)

print("Train 정확도 :", round(train_acc, 4))
print("Test 정확도  :", round(test_acc, 4))


# 과적합 판단
gap = train_acc - test_acc

print("\n성능 차이 :", round(gap, 4))

if gap > 0.1:
    print("→ 과적합 의심")
elif gap > 0.03:
    print("→ 약간 과적합 가능")
else:
    print("→ 과적합 아님")


# 7 시각화
plt.figure(figsize=(12,8))
plot_tree(
    dt,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True
)

plt.show()